<a href="https://www.kaggle.com/code/avikdas567/neural-program-synthesis-for-arc-via-onnx-models?scriptVersionId=312295258" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

## 🧠 Neural Program Synthesis for ARC via ONNX Models

The ARC-AGI benchmark challenges models to generalize from just a few examples. This is a setting where conventional deep learning approaches often struggle. Instead of training large models, this notebook explores a different direction: **constructing task-specific neural programs directly as ONNX graphs**.

For each task, we infer the underlying transformation (e.g., color mapping, structural patterns, or object isolation) and encode it into a minimal neural network. This reframes each ARC problem as a **program synthesis task**, where the “program” is represented as a deterministic neural architecture.

This perspective suggests a key insight:

> Solving ARC may be less about scaling models, and more about **building the right computation for each task**.

## 🔍 Key Ideas
- Treat ARC tasks as **program synthesis problems**
- Encode transformations as **ONNX computation graphs**
- Use **deterministic reasoning** instead of training
- Construct **task-specific neural networks**

## Imports

In [1]:
import os
import json
import numpy as np
from tqdm import tqdm
import zipfile

import onnx
from onnx import helper, TensorProto

## Paths

In [2]:
BASE_PATH = "/kaggle/input/competitions/neurogolf-2026"
TASK_FILES = sorted([f for f in os.listdir(BASE_PATH) if f.endswith(".json")])

print("Total tasks:", len(TASK_FILES))

Total tasks: 400


## Load + Utils

In [3]:
def load_task(path):
    with open(path, "r") as f:
        return json.load(f)

In [4]:
def finalize_model(graph):
    model = helper.make_model(
        graph,
        producer_name="neurogolf",
        opset_imports=[helper.make_opsetid("", 13)]
    )
    
    model.ir_version = 7
    
    return model

## Transformation Detection

In [5]:
def detect_transformation(task):
    train = task["train"]

    def to_np(x):
        return np.array(x)

    # Identity
    if all(pair["input"] == pair["output"] for pair in train):
        return ("identity", None)

    # Color mapping
    mapping = {}
    valid = True
    for pair in train:
        inp = to_np(pair["input"])
        out = to_np(pair["output"])
        if inp.shape != out.shape:
            valid = False
            break
        for i in range(inp.shape[0]):
            for j in range(inp.shape[1]):
                if inp[i,j] not in mapping:
                    mapping[inp[i,j]] = out[i,j]
                elif mapping[inp[i,j]] != out[i,j]:
                    valid = False
                    break
    if valid:
        return ("color_map", mapping)

    # Rotations
    for k in [1,2,3]:
        if all(np.array_equal(np.rot90(to_np(p["input"]), k=k), to_np(p["output"])) for p in train):
            return ("rot", k)

    # Flips
    if all(np.array_equal(np.fliplr(to_np(p["input"])), to_np(p["output"])) for p in train):
        return ("flip_lr", None)

    if all(np.array_equal(np.flipud(to_np(p["input"])), to_np(p["output"])) for p in train):
        return ("flip_ud", None)

    # Crop
    def crop(x):
        x = np.array(x)
        ys, xs = np.where(x != 0)
        if len(ys) == 0:
            return x
        return x[min(ys):max(ys)+1, min(xs):max(xs)+1]

    if all(np.array_equal(crop(p["input"]), np.array(p["output"])) for p in train):
        return ("crop", None)

    # Object isolate
    def isolate(x):
        x = np.array(x)
        vals, counts = np.unique(x[x != 0], return_counts=True)
        if len(vals) == 0:
            return x
        main = vals[np.argmax(counts)]
        return (x == main).astype(int) * main

    if all(np.array_equal(isolate(p["input"]), np.array(p["output"])) for p in train):
        return ("isolate", None)

    return ("unknown", None)

## MODEL BUILDERS

In [6]:
def build_identity_model(task_id):
    input_name = "input"
    output_name = "output"

    node = helper.make_node("Identity", [input_name], [output_name])

    graph = helper.make_graph(
        [node],
        f"task{task_id}",
        [helper.make_tensor_value_info(input_name, TensorProto.FLOAT, [1,10,30,30])],
        [helper.make_tensor_value_info(output_name, TensorProto.FLOAT, [1,10,30,30])]
    )

    return finalize_model(graph)


def build_color_map_model(task_id, mapping):
    input_name = "input"
    output_name = "output"
    weight_name = "W"

    weight = np.zeros((10,10,1,1), dtype=np.float32)
    for i in range(10):
        if i in mapping:
            weight[mapping[i], i, 0, 0] = 1.0

    W_tensor = helper.make_tensor(
        name=weight_name,
        data_type=TensorProto.FLOAT,
        dims=weight.shape,
        vals=weight.flatten().astype(np.float32),
    )

    node = helper.make_node(
        "Conv",
        inputs=[input_name, weight_name],
        outputs=[output_name],
        kernel_shape=[1,1]
    )

    graph = helper.make_graph(
        [node],
        f"task{task_id}",
        inputs=[
            helper.make_tensor_value_info(input_name, TensorProto.FLOAT, [1,10,30,30])
        ],
        outputs=[
            helper.make_tensor_value_info(output_name, TensorProto.FLOAT, [1,10,30,30])
        ],
        initializer=[W_tensor]
    )

    return finalize_model(graph)


def build_mask_model(task_id, color):
    input_name = "input"
    output_name = "output"
    weight_name = "W"

    weight = np.zeros((10,10,1,1), dtype=np.float32)
    weight[color, color, 0, 0] = 1.0

    W_tensor = helper.make_tensor(
        name=weight_name,
        data_type=TensorProto.FLOAT,
        dims=weight.shape,
        vals=weight.flatten().astype(np.float32),
    )

    node = helper.make_node(
        "Conv",
        inputs=[input_name, weight_name],
        outputs=[output_name],
        kernel_shape=[1,1]
    )

    graph = helper.make_graph(
        [node],
        f"task{task_id}",
        inputs=[
            helper.make_tensor_value_info(input_name, TensorProto.FLOAT, [1,10,30,30])
        ],
        outputs=[
            helper.make_tensor_value_info(output_name, TensorProto.FLOAT, [1,10,30,30])
        ],
        initializer=[W_tensor]
    )

    return finalize_model(graph)

## SOLVER

In [7]:
def solve_task(task, task_id):
    t, param = detect_transformation(task)

    if t == "identity":
        return build_identity_model(task_id)

    if t == "color_map":
        return build_color_map_model(task_id, param)

    if t == "isolate":
        vals = []
        for p in task["train"]:
            x = np.array(p["input"])
            v, c = np.unique(x[x != 0], return_counts=True)
            if len(v):
                vals.append(v[np.argmax(c)])
        if len(vals):
            color = max(set(vals), key=vals.count)
            return build_mask_model(task_id, color)

    # safe fallback
    return build_identity_model(task_id)

## GENERATE MODELS

In [8]:
OUTPUT_DIR = "/kaggle/working/models"
os.makedirs(OUTPUT_DIR, exist_ok=True)

for idx, file in enumerate(tqdm(TASK_FILES)):
    task = load_task(os.path.join(BASE_PATH, file))
    model = solve_task(task, idx+1)

    onnx.save(model, os.path.join(OUTPUT_DIR, f"task{idx+1:03d}.onnx"))

100%|██████████| 400/400 [00:07<00:00, 50.72it/s]


## VALIDATE

In [9]:
for f in os.listdir(OUTPUT_DIR):
    model = onnx.load(os.path.join(OUTPUT_DIR, f))
    onnx.checker.check_model(model)

print("All models valid ✅")

All models valid ✅


In [10]:
for f in os.listdir(OUTPUT_DIR):
    path = os.path.join(OUTPUT_DIR, f)
    try:
        model = onnx.load(path)
        onnx.checker.check_model(model)
    except Exception as e:
        print("FAILED:", f, e)

## ZIP SUBMISSION

In [11]:
zip_path = "/kaggle/working/submission.zip"

with zipfile.ZipFile(zip_path, 'w') as z:
    for f in os.listdir(OUTPUT_DIR):
        z.write(os.path.join(OUTPUT_DIR, f), f)

print("Submission saved:", zip_path)

Submission saved: /kaggle/working/submission.zip
